# 11 — Ingest Pipeline

Tests `services/ingest_to_qdrant.py`:
- `load_catalog()` reads and validates the JSON file
- `build_text()` creates a non-empty, meaningful text blob per product
- `embed_products()` returns the right shape
- `build_points()` creates valid PointStruct objects
- Full `run_ingest()` pipeline runs without errors (idempotent re-upsert)

> **Requires**: Qdrant credentials. Does NOT re-ingest from scratch (uses `recreate=False`).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv
load_dotenv(os.path.join('..', '.env'))

In [ ]:
from services.ingest_to_qdrant import load_catalog, build_text, embed_products, build_points
from utils.config import QDRANT_EMBEDDING_DIM, CATALOG_PATH

## 1. load_catalog() — data file integrity

In [ ]:
products = load_catalog()

print(f'Products loaded: {len(products):,}')
print('First product:')
import json
print(json.dumps(products[0], indent=2))

assert isinstance(products, list), 'load_catalog must return a list'
assert len(products) > 1000, f'Expected > 1000 products, got {len(products)}'

required_keys = {'name', 'price', 'category', 'product_url'}
for i, p in enumerate(products[:10]):  # Spot-check first 10
    missing = required_keys - set(p.keys())
    assert not missing, f'Product {i} missing keys: {missing}'

print('load_catalog: PASSED')

## 2. Category distribution

In [ ]:
from collections import Counter

products = load_catalog()
category_counts = Counter(p.get('category', 'unknown') for p in products)

print('Category distribution:')
for cat, count in sorted(category_counts.items(), key=lambda x: -x[1]):
    print(f'  {cat:15s}: {count:,}')

expected_categories = {'cakes', 'flowers', 'books', 'fashion', 'gifts', 'vouchers', 'electronics'}
assert expected_categories.issubset(category_counts.keys()), \
    f'Missing categories: {expected_categories - set(category_counts.keys())}'

print('Category distribution: PASSED')

## 3. build_text() — text blob quality

In [ ]:
products = load_catalog()

# Test on a sample of products
for product in products[:10]:
    text = build_text(product)
    assert isinstance(text, str), f'build_text must return str, got {type(text)}'
    assert len(text) > 10, f'Text too short for: {product["name"]}'
    # Key fields should appear in the text
    assert product['name'].lower() in text.lower() or len(text) > 20, \
        f'Product name not in text blob: {product["name"]}'

# Show a sample
print('Sample text blob:')
print(build_text(products[0]))

print('\nbuild_text: PASSED')

## 4. embed_products() — shape check on a small batch

In [ ]:
products = load_catalog()[:5]  # Use only 5 products for speed
texts = [build_text(p) for p in products]

embeddings = embed_products(texts)

print(f'Embeddings shape: {len(embeddings)} x {len(embeddings[0])}')

assert len(embeddings) == len(texts), f'Expected {len(texts)} embeddings, got {len(embeddings)}'
assert len(embeddings[0]) == QDRANT_EMBEDDING_DIM, \
    f'Expected dim {QDRANT_EMBEDDING_DIM}, got {len(embeddings[0])}'
assert all(isinstance(v, float) for v in embeddings[0][:5]), 'Values must be floats'

print('embed_products: PASSED')

## 5. build_points() — PointStruct validation

In [ ]:
from qdrant_client.models import PointStruct

products = load_catalog()[:3]
texts = [build_text(p) for p in products]
embeddings = embed_products(texts)
points = build_points(products, embeddings)

print(f'Points created: {len(points)}')
print('First point:')
print(f'  id     : {points[0].id}')
print(f'  vector : [{points[0].vector[0]:.4f}, ...] (len={len(points[0].vector)})')
print(f'  payload: {list(points[0].payload.keys())}')

assert len(points) == len(products)
for i, pt in enumerate(points):
    assert isinstance(pt, PointStruct), f'Point {i} is not a PointStruct'
    assert len(pt.vector) == QDRANT_EMBEDDING_DIM
    assert 'name' in pt.payload
    assert 'category' in pt.payload

print('build_points: PASSED')

## 6. run_ingest() is idempotent (no recreate)

In [ ]:
from services.ingest_to_qdrant import run_ingest
from infrastructure.db.qdrant_store import collection_info

info_before = collection_info()
print(f'Points before: {info_before["points_count"]:,}')

# Re-upsert without recreating — should be safe and idempotent
run_ingest(recreate=False)

info_after = collection_info()
print(f'Points after : {info_after["points_count"]:,}')

assert info_after['points_count'] == info_before['points_count'], \
    'Point count should not change on idempotent re-upsert'
print('run_ingest idempotent: PASSED')

## 7. Custom catalog path

In [ ]:
# Test loading a specific category file instead of the full catalog
import os

cakes_path = os.path.join('..', 'data', 'cakes.json')
if os.path.exists(cakes_path):
    cakes = load_catalog(path=cakes_path)
    print(f'Cakes loaded: {len(cakes)}')
    assert all(p.get('category') == 'cakes' for p in cakes), \
        'All products in cakes.json should have category=cakes'
    print('Custom catalog path: PASSED')
else:
    print('cakes.json not found at expected path — skipping')